In [1]:
import pandas as pd

Databases

In [2]:
orders = '../databases/orders.csv'

Read csv

In [3]:
orders_df_raw = pd.read_csv(orders)
orders_df = orders_df_raw.copy()

In [4]:
orders_df["created_at"] = pd.to_datetime(
    orders_df["created_at"],
    format="ISO8601"
)

cohort = (
    orders_df.groupby("user_id")["created_at"]
    .min()
    .dt.to_period("M")
    .dt.to_timestamp()
    .reset_index(name="cohort_month")
)
cohort

C:\Users\User\AppData\Local\Temp\ipykernel_7464\3222126526.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


,user_id,cohort_month
0,2,2026-05-01
1,5,2026-03-01
2,6,2024-04-01
3,7,2022-01-01
4,8,2023-07-01
...,...,...
80057,99996,2026-03-01
80058,99997,2022-05-01
80059,99998,2022-01-01
80060,99999,2022-06-01


In [5]:
activity = (
    orders_df.assign(
        activity_month=orders_df["created_at"]
        .dt.to_period("M")
        .dt.to_timestamp()
    )
    [["user_id", "activity_month"]]
    .drop_duplicates()
)
activity

C:\Users\User\AppData\Local\Temp\ipykernel_7464\1927387750.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


,user_id,activity_month
0,8,2023-07-01
1,29,2026-02-01
2,59,2025-06-01
3,62,2026-05-01
4,63,2023-11-01
...,...,...
125417,99983,2025-12-01
125418,99987,2026-03-01
125419,99997,2022-05-01
125420,99997,2025-02-01


In [6]:
cohort_size = (
    cohort.groupby("cohort_month")["user_id"]
    .count()
    .reset_index(name="cohort_users")
)
cohort_size

,cohort_month,cohort_users
0,2019-01-01,8
1,2019-02-01,18
2,2019-03-01,39
3,2019-04-01,68
4,2019-05-01,76
...,...,...
84,2026-01-01,2297
85,2026-02-01,2190
86,2026-03-01,2645
87,2026-04-01,2882


In [7]:
retention = (
    cohort.merge(activity, on="user_id")
    .merge(cohort_size, on="cohort_month")
)

# months difference
retention["months_since_first"] = (
    (retention["activity_month"].dt.year - retention["cohort_month"].dt.year) * 12
    + (retention["activity_month"].dt.month - retention["cohort_month"].dt.month)
)

# filter conditions
retention = retention[
    (retention["months_since_first"].between(0, 11)) &
    (
        retention["cohort_month"] >=
        (pd.Timestamp.today().normalize() - pd.DateOffset(months=18))
    )
]

# final aggregation
retention = (
    retention.groupby(
        ["cohort_month", "cohort_users", "months_since_first"]
    )["user_id"]
    .nunique()
    .reset_index(name="retained_users")
)

# retention %
retention["retention_rate_pct"] = round(
    retention["retained_users"] * 100 / retention["cohort_users"],
    1
)

# order by
retention = retention.sort_values(
    ["cohort_month", "months_since_first"]
)

retention

,cohort_month,cohort_users,months_since_first,retained_users,retention_rate_pct
0,2024-12-01,1405,0,1405,100.0
1,2024-12-01,1405,1,40,2.8
2,2024-12-01,1405,2,57,4.1
3,2024-12-01,1405,3,43,3.1
4,2024-12-01,1405,4,39,2.8
...,...,...,...,...,...
145,2026-03-01,2645,1,314,11.9
146,2026-03-01,2645,2,263,9.9
147,2026-04-01,2882,0,2882,100.0
148,2026-04-01,2882,1,459,15.9


Retention rate cannot exceed 100%

In [8]:
retention["retention_rate_pct"].max()

np.float64(100.0)